# Workshop: Single Tool SRE Agent

## Overview

In this workshop module, you'll build a Strands Agent with one tool that can investigate Kubernetes pod failures. This demonstrates the core concept of AI-powered infrastructure troubleshooting.

### Learning Objectives

By the end of this module, you will:
- Understand how Strands Agents work with @tool decorators
- Create a FastAPI backend with realistic infrastructure data
- Build an AI agent that can investigate and solve infrastructure problems
- See the value proposition of AI-powered SRE automation

### Prerequisites

- AWS Account with Amazon Bedrock access
- Claude 3 Haiku model enabled in your AWS account
- Python 3.9+ environment

### Architecture

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│                 │    │                 │    │                 │
│ Strands Agent   │───▶│ @tool Function  │───▶│ FastAPI Backend │
│                 │    │                 │    │                 │
│ • Claude Haiku  │    │ get_pod_status  │    │ • Pod Data      │
│ • Investigation │    │                 │    │ • Realistic API │
└─────────────────┘    └─────────────────┘    └─────────────────┘
```

**Estimated completion time:** 15 minutes

## Step 1: Environment Setup

Install the required packages for this workshop module.

In [ ]:
%%bash
pip install fastapi uvicorn strands requests --quiet
echo "✅ Packages installed successfully"

In [7]:
# Import required libraries
from fastapi import FastAPI
from strands import Agent, tool
from strands.models import BedrockModel
import uvicorn
import threading
import time
import requests

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## Step 2: Create Infrastructure Backend

Create a FastAPI backend that simulates a Kubernetes cluster with realistic pod data.

In [9]:
# Create FastAPI application with realistic Kubernetes data
app = FastAPI(title="Kubernetes API Simulator", version="1.0.0")

# Realistic pod data representing a failure scenario
PODS_DATA = {
    "pods": [
        {
            "name": "payment-service-7d4f8-x5m1q",
            "namespace": "production",
            "status": "CrashLoopBackOff",
            "ready": False,
            "restart_count": 15,
            "cpu_usage": "25%",
            "memory_usage": "98%",
            "node": "worker-node-2",
            "last_restart": "2024-01-15T14:24:30Z",
            "containers": [
                {
                    "name": "payment-api",
                    "image": "payment-service:v1.2.3",
                    "status": "Waiting",
                    "reason": "CrashLoopBackOff",
                    "message": "Back-off 5m0s restarting failed container"
                }
            ],
            "events": [
                "Warning: OutOfMemoryError in container payment-api",
                "Warning: Back-off restarting failed container",
                "Error: Container payment-api failed with exit code 137"
            ]
        },
        {
            "name": "user-service-9k2x1-y6n2r",
            "namespace": "production",
            "status": "Running",
            "ready": True,
            "restart_count": 0,
            "cpu_usage": "32%",
            "memory_usage": "64%",
            "node": "worker-node-1",
            "last_restart": None,  # Changed from null to None
            "containers": [
                {
                    "name": "user-api",
                    "image": "user-service:v1.1.0",
                    "status": "Running",
                    "reason": "Started",
                    "message": "Container started successfully"
                }
            ],
            "events": [
                "Normal: Successfully pulled image user-service:v1.1.0",
                "Normal: Created container user-api",
                "Normal: Started container user-api"
            ]
        }
    ]
}

@app.get("/health")
def health_check():
    """Health check endpoint"""
    return {"status": "healthy", "service": "kubernetes-api"}

@app.get("/pods")
def get_pods():
    """Get all pods in the cluster"""
    return PODS_DATA

print("✅ FastAPI backend created with realistic pod data")

✅ FastAPI backend created with realistic pod data


In [10]:
# Start the FastAPI server in background
def start_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="error")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Wait for server startup
time.sleep(3)

# Verify server is running
try:
    response = requests.get("http://127.0.0.1:8000/health", timeout=5)
    if response.status_code == 200:
        print("✅ Backend server running at http://127.0.0.1:8000")
        print(f"   Health status: {response.json()['status']}")
    else:
        print(f"❌ Server health check failed: {response.status_code}")
except Exception as e:
    print(f"❌ Cannot connect to server: {e}")

✅ Backend server running at http://127.0.0.1:8000
   Health status: healthy


## Step 3: Create Strands Agent Tool

Define a single tool that the Strands Agent can use to investigate pod issues.

In [11]:
@tool
def get_pod_status(namespace: str = "production") -> str:
    """
    Get detailed status information for Kubernetes pods in the specified namespace.
    
    Args:
        namespace: Kubernetes namespace to query (default: production)
        
    Returns:
        Comprehensive pod status including health, resource usage, and recent events
    """
    try:
        response = requests.get("http://127.0.0.1:8000/pods", timeout=10)
        response.raise_for_status()
        data = response.json()
        
        # Filter pods by namespace
        filtered_pods = [pod for pod in data["pods"] if pod["namespace"] == namespace]
        
        if not filtered_pods:
            return f"No pods found in namespace '{namespace}'"
        
        result = f"Found {len(filtered_pods)} pods in '{namespace}' namespace:\n\n"
        
        for pod in filtered_pods:
            status_icon = "❌" if not pod["ready"] else "✅"
            result += f"{status_icon} Pod: {pod['name']}\n"
            result += f"   Status: {pod['status']} (Ready: {pod['ready']})\n"
            result += f"   Restarts: {pod['restart_count']}\n"
            result += f"   Resource Usage: CPU {pod['cpu_usage']}, Memory {pod['memory_usage']}\n"
            result += f"   Node: {pod['node']}\n"
            
            if pod.get('last_restart'):
                result += f"   Last Restart: {pod['last_restart']}\n"
            
            # Include container details
            if pod.get('containers'):
                result += f"   Containers:\n"
                for container in pod['containers']:
                    result += f"     - {container['name']}: {container['status']} ({container['reason']})\n"
            
            # Include recent events
            if pod.get('events'):
                result += f"   Recent Events:\n"
                for event in pod['events'][-3:]:  # Show last 3 events
                    result += f"     - {event}\n"
            
            result += "\n"
            
        return result
        
    except requests.RequestException as e:
        return f"Error querying Kubernetes API: {e}"
    except Exception as e:
        return f"Unexpected error: {e}"

print("✅ Strands tool function created: get_pod_status()")

✅ Strands tool function created: get_pod_status()


## Step 4: Initialize Strands Agent

Create a Strands Agent using Amazon Bedrock Claude 3 Haiku model.

In [12]:
# Initialize Bedrock model and Strands Agent
try:
    # Create Bedrock model instance
    model = BedrockModel(model_id="us.anthropic.claude-3-haiku-20240307-v1:0")
    
    # Create Strands Agent with professional SRE system prompt
    agent = Agent(
        model=model,
        tools=[get_pod_status],
        system_prompt="""You are an expert Site Reliability Engineer (SRE) investigating infrastructure issues.

        Your responsibilities:
        1. Use available tools to systematically gather infrastructure data
        2. Analyze the information to identify root causes and contributing factors
        3. Provide specific, actionable recommendations for resolution
        4. Explain your reasoning clearly and concisely
        5. Focus on immediate fixes and preventive measures

        When investigating issues:
        - Look for patterns in pod restarts, resource usage, and error messages
        - Consider both immediate symptoms and underlying causes
        - Provide confidence levels for your assessments
        - Recommend specific commands or actions to resolve issues
        - Suggest monitoring improvements to prevent future incidents

        Be direct, technical, and solution-focused in your analysis."""
    )
    
    print("✅ Strands Agent initialized successfully")
    print(f"   Model: Claude 3 Haiku (us.anthropic.claude-3-haiku-20240307-v1:0)")
    print(f"   Tools: 1 tool available (get_pod_status)")
    print(f"   Framework: Strands Agents")
    
except Exception as e:
    print(f"❌ Failed to initialize Strands Agent: {e}")
    print("\nTroubleshooting steps:")
    print("1. Verify AWS credentials: aws configure list")
    print("2. Check Bedrock access in your AWS region")
    print("3. Ensure Claude 3 Haiku model is enabled")
    agent = None

✅ Strands Agent initialized successfully
   Model: Claude 3 Haiku (us.anthropic.claude-3-haiku-20240307-v1:0)
   Tools: 1 tool available (get_pod_status)
   Framework: Strands Agents


## Step 5: Run Investigation

Let the Strands Agent investigate a critical production incident.

In [13]:
if agent:
    print("🚨 PRODUCTION INCIDENT")
    print("=" * 40)
    print("ALERT: Payment service is down!")
    print("Impact: Users cannot complete purchases")
    print("Priority: P1 - Critical")
    print("\nStarting AI investigation...\n")
    
    # Record investigation start time
    start_time = time.time()
    
    # Run the investigation
    incident_description = (
        "URGENT: Payment service appears to be down in production. "
        "Users are reporting they cannot complete purchases. "
        "Please investigate the production environment immediately and "
        "identify what's wrong with the payment service."
    )
    
    try:
        response = agent(incident_description)
        investigation_time = round(time.time() - start_time, 1)
        
        print(f"⚡ Investigation completed in {investigation_time} seconds")
        print("\n" + "=" * 60)
        print("AI INVESTIGATION RESULTS")
        print("=" * 60)
        
        # Access the response correctly
        if hasattr(response, 'content'):
            print(response.content)
        elif hasattr(response, 'message'):
            print(response.message)
        else:
            print(str(response))
        
        print("\n" + "=" * 60)
        
    except Exception as e:
        print(f"❌ Investigation failed: {e}")
        
else:
    print("❌ Cannot run investigation - Strands Agent not initialized")
    print("Please check the previous steps for errors")

🚨 PRODUCTION INCIDENT
ALERT: Payment service is down!
Impact: Users cannot complete purchases
Priority: P1 - Critical

Starting AI investigation...


Tool #1: get_pod_status


The investigation indicates that the payment-service pod is in a CrashLoopBackOff state, meaning it is repeatedly crashing and being restarted by the Kubernetes cluster. This is likely the root cause of the payment service being unavailable to users.

The key evidence is:
- The payment-service pod has restarted 15 times in a short period of time
- It's running out of memory (98% usage) and experiencing an OutOfMemoryError
- The container is stuck in a "Waiting" state with a CrashLoopBackOff error

This suggests a critical issue with the payment-service application, likely a memory leak or other resource exhaustion problem. The user-service pod appears to be running normally, so the issue is isolated to the payment-service component.

To resolve this incident, I recommend the following:

1. [95% confidence] Scale 

## Step 6: Review Results

Analyze what the Strands Agent accomplished.

In [14]:
if agent and 'response' in locals():
    print("📊 INVESTIGATION ANALYSIS")
    print("=" * 30)
    
    # Show what the agent accomplished
    print("\nWhat the AI Agent did:")
    print("✅ Automatically selected the appropriate tool")
    print("✅ Gathered comprehensive pod status information")
    print("✅ Analyzed the data to identify root cause")
    print("✅ Provided specific resolution steps")
    print("✅ Suggested preventive measures")
    
    # Compare to manual investigation
    print("\nComparison to Manual Investigation:")
    print(f"⚡ AI Investigation: {investigation_time} seconds")
    print("⏳ Manual Investigation: 15-30 minutes typical")
    print("📈 Speed Improvement: ~95% faster")
    
    print("\nKey Benefits Demonstrated:")
    print("• Instant infrastructure analysis")
    print("• Consistent investigation methodology")
    print("• Professional SRE-level reasoning")
    print("• 24/7 availability for incident response")
    
else:
    print("Investigation results not available")
    print("Please ensure the previous steps completed successfully")

📊 INVESTIGATION ANALYSIS

What the AI Agent did:
✅ Automatically selected the appropriate tool
✅ Gathered comprehensive pod status information
✅ Analyzed the data to identify root cause
✅ Provided specific resolution steps
✅ Suggested preventive measures

Comparison to Manual Investigation:
⚡ AI Investigation: 7.1 seconds
⏳ Manual Investigation: 15-30 minutes typical
📈 Speed Improvement: ~95% faster

Key Benefits Demonstrated:
• Instant infrastructure analysis
• Consistent investigation methodology
• Professional SRE-level reasoning
• 24/7 availability for incident response


## Summary and Next Steps

### What You Accomplished

In this workshop module, you:

1. **Created a FastAPI backend** with realistic Kubernetes pod data
2. **Built a Strands Agent** using Amazon Bedrock Claude 3 Haiku
3. **Defined a custom @tool** that can query infrastructure APIs
4. **Demonstrated AI-powered incident response** in seconds vs. minutes

### Key Learnings

- **Strands Agents** provide a simple way to create AI agents with tool access
- **@tool decorators** make any Python function accessible to AI agents
- **Claude 3 Haiku** offers cost-effective yet capable AI reasoning
- **Infrastructure automation** can dramatically reduce incident response time

### Workshop Progression

This module demonstrated the core concept with one tool. The complete workshop series will show you how to:

- **Module 1**: Multiple tools with single agent (3 Kubernetes tools)
- **Module 2**: Secure gateway integration with MCP protocol  
- **Module 3**: Multi-agent architecture with specialist agents
- **Module 4**: Memory integration for persistent learning
- **Module 5**: Production deployment to AgentCore Runtime

### Resources

- [Strands Documentation](https://strands.dev)
- [Amazon Bedrock User Guide](https://docs.aws.amazon.com/bedrock/)
- [FastAPI Documentation](https://fastapi.tiangolo.com/)

---

**Next**: Continue to Module 1 to expand your agent with multiple tools and more complex scenarios.